## TL;DR — Plain English

**What this notebook does:**
Takes you from zero knowledge to a working mental model of AlphaFold 3 in 3 hours.

**By the end, you will understand:**
- What a protein is (amino acid sequence → 3D structure)
- Why predicting the 3D structure from sequence was hard for 60 years
- The complete AF3 architecture: what goes in, what comes out, what each component does
- Why triangle attention (not regular attention) is needed for 3D structure
- How diffusion generates structures from noise
- The three equations that define AF3 mathematically

**You do NOT need to know:** biology, chemistry, structural biology, or advanced math.
You need: Python loops and functions, basic matrix multiply intuition.

**After this notebook:** Go to `07/01_af3_architecture.ipynb` for implementation details.


# AlphaFold 3 — Zero to Hero
## Start Here If You Have No Background

This notebook assumes you know **basic Python** but nothing about proteins, biology, or structure prediction.
By the end you will understand the entire AlphaFold 3 problem from the ground up.

**Who this is for:** Anyone starting modules 07 or 10 who finds the other notebooks assume too much.

**Time:** ~3 hours | **No prerequisites beyond Python lists and loops**


---
## Chapter 1: What Is a Protein? (Biology in 10 Minutes)

A protein is a **string of amino acids** that folds into a specific 3D shape.
That shape determines what the protein DOES in your body.

```
Amino acid string: M-V-H-L-T-P-E-E-K-S-A-V-T...
       ↓ (folds spontaneously in milliseconds)
3D structure: [complex tangled shape]
       ↓
Function: carries oxygen (hemoglobin), catalyzes reactions, builds cells
```

**There are only 20 standard amino acids.** Each has a 1-letter code (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y).


In [ ]:
# The 20 amino acids — learn these, they appear everywhere in AF3 code
amino_acids = {
    'A': ('Alanine',       'nonpolar',  'small',     'CHₓ side chain'),
    'C': ('Cysteine',      'nonpolar',  'small',     'SH — forms disulfide bonds'),
    'D': ('Aspartate',     'charged-',  'small',     'COO⁻ — negative at pH 7'),
    'E': ('Glutamate',     'charged-',  'medium',    'COO⁻ — negative, longer'),
    'F': ('Phenylalanine', 'aromatic',  'large',     'benzyl ring'),
    'G': ('Glycine',       'nonpolar',  'tiny',      'no side chain — max flexibility'),
    'H': ('Histidine',     'aromatic',  'medium',    'pKa≈6, switches charge at pH 6-8'),
    'I': ('Isoleucine',    'nonpolar',  'large',     'branched hydrophobic'),
    'K': ('Lysine',        'charged+',  'long',      'NH₃⁺ — positive'),
    'L': ('Leucine',       'nonpolar',  'large',     'branched hydrophobic'),
    'M': ('Methionine',    'nonpolar',  'medium',    'thioether — start codon AUG'),
    'N': ('Asparagine',    'polar',     'small',     'amide — hydrogen bonds'),
    'P': ('Proline',       'nonpolar',  'rigid',     'ring — breaks helix'),
    'Q': ('Glutamine',     'polar',     'medium',    'amide — hydrogen bonds, longer'),
    'R': ('Arginine',      'charged+',  'long',      'guanidinium — strongest +charge'),
    'S': ('Serine',        'polar',     'small',     'OH — phosphorylation target'),
    'T': ('Threonine',     'polar',     'medium',    'OH — phosphorylation target'),
    'V': ('Valine',        'nonpolar',  'medium',    'branched hydrophobic'),
    'W': ('Tryptophan',    'aromatic',  'largest',   'indole ring — fluorescent'),
    'Y': ('Tyrosine',      'aromatic',  'large',     'phenol — tyrosine kinase target'),
}

# In AF3 code, amino acids are encoded as integers 0-19 + special tokens
AA_TO_INT = {aa: i for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
INT_TO_AA = {v: k for k, v in AA_TO_INT.items()}

print("The 20 amino acids and their properties:")
print(f"{'AA'} {'Name':<15} {'Class':<12} {'Size':<8} {'Key feature'}")
print("-" * 65)
for aa, (name, cls, size, feat) in amino_acids.items():
    print(f" {aa}  {name:<15} {cls:<12} {size:<8} {feat}")

print(f"
AA_TO_INT encoding used in PyTorch:")
print("  'A' → 0, 'C' → 1, 'D' → 2 ...")
print(f"  Full alphabet: {AA_TO_INT}")
print()

# Key insight: hydrophobic AAs go INSIDE (away from water)
#              charged/polar AAs go OUTSIDE (interact with water)
hydrophobic = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if cls in ('nonpolar', 'aromatic')]
charged     = [aa for aa, (_, cls, _, _) in amino_acids.items()
               if 'charged' in cls or cls == 'polar']
print(f"Hydrophobic (prefer inside): {hydrophobic}")
print(f"Polar/Charged (prefer surface): {charged}")
print()
print("WHY THIS MATTERS FOR AF3:")
print("  Hydrophobic collapse = primary driving force of protein folding")
print("  AF3 learns these preferences from millions of known structures")

> 📋 **Expected output:** A dictionary or table of the 20 amino acids with their one-letter codes, three-letter codes, and properties.
> ✅ You should see 20 entries. If you see an error, check that the cell ran completely.